# Challenge Locaweb EDA

## Configurações Iniciais

### Configurações a depender do ambiente

In [1]:
import os
import sys
import subprocess

# --- CONFIGURAÇÃO ALVO ---
TARGET_PYSPARK = "4.1.1"

# 1. Identifica o Ambiente (Fora da função para ser global)
IN_COLAB = 'google.colab' in sys.modules
ENV_NAME = "☁️ Google Colab" if IN_COLAB else "💻 Ambiente Local (WSL/Jupyter)"

print(f"Detectado: {ENV_NAME}")
print(f"Versão do Python: {sys.version.split()[0]}")

# 2. Define os Caminhos Globais
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = "/content/drive/MyDrive/fiap/segundo ano/challenge_locaweb/"
else:
    BASE_PATH = "raw data/" # Adicionei a barra no final para facilitar o join

def setup_pyspark():
    if IN_COLAB:
        try:
            import pyspark
            if pyspark.__version__ != TARGET_PYSPARK:
                print(f"(!) Atualizando PySpark de {pyspark.__version__} para {TARGET_PYSPARK}...")
                subprocess.check_call([sys.executable, "-m", "pip", "install", f"pyspark=={TARGET_PYSPARK}", "-q"])
                print("🚨 Reinicie o Ambiente (Runtime > Restart Session) para aplicar a mudança!")
        except ImportError:
            print(f"(!) Instalando PySpark {TARGET_PYSPARK}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", f"pyspark=={TARGET_PYSPARK}", "-q"])

    # Verificação Final
    try:
        import pyspark
        if pyspark.__version__ == TARGET_PYSPARK:
            print(f"✅ PySpark {pyspark.__version__} pronto!")
        else:
            print(f"⚠️ Alerta: PySpark está na versão {pyspark.__version__}. Alvo era {TARGET_PYSPARK}.")
    except ImportError:
        print("❌ Erro: PySpark não encontrado.")

setup_pyspark()

Detectado: 💻 Ambiente Local (WSL/Jupyter)
Versão do Python: 3.12.3
✅ PySpark 4.1.1 pronto!


### Importação de Bibliotecas

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql.types import * # Para definir Schemas (StructType, DoubleType, etc)
from pyspark.sql.window import Window
import pandas as pd

### Inicialização da Sessão Spark

In [3]:
spark = SparkSession.builder \
    .appName("DataExploration") \
    .master("local[*]") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/02 00:10:43 WARN Utils: Your hostname, PCJULIA, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/02 00:10:43 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/02 00:10:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
spark

### Importação de Dados

In [5]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", ";") \
    .option("encoding", "ISO-8859-1") \
    .load(f"{BASE_PATH}/LW-DATASET-CSV.CSV")

In [6]:
df.show(20)

+----------+----------+-------+---------+------------+---------------+--------------------+-------------------+---------+-------------------+-------+--------------------+--------------------+-------+-------------+-------------+---------------+----------------+------------+
|    Número|Prioridade|Produto|Categoria|Subcategoria|Grupo designado|Item de configuração|             Aberto|Resolvido|          Encerrado|Duração|Código de fechamento|  Descrição resumida|Solução|   Aberto por|Incidente Pai|         Status|Entrou para KPI?|KPI Violado?|
+----------+----------+-------+---------+------------+---------------+--------------------+-------------------+---------+-------------------+-------+--------------------+--------------------+-------+-------------+-------------+---------------+----------------+------------+
|INC8654273| 3 - Média|   NULL|     NULL|        NULL|         Team14|             IC00001|2025-12-31 23:45:18|     NULL|2025-12-31 23:45:32|     14|                NULL|Problem:

In [7]:
df.printSchema()

root
 |-- Número: string (nullable = true)
 |-- Prioridade: string (nullable = true)
 |-- Produto: string (nullable = true)
 |-- Categoria: string (nullable = true)
 |-- Subcategoria: string (nullable = true)
 |-- Grupo designado: string (nullable = true)
 |-- Item de configuração: string (nullable = true)
 |-- Aberto: timestamp (nullable = true)
 |-- Resolvido: timestamp (nullable = true)
 |-- Encerrado: timestamp (nullable = true)
 |-- Duração: integer (nullable = true)
 |-- Código de fechamento: string (nullable = true)
 |-- Descrição resumida: string (nullable = true)
 |-- Solução: string (nullable = true)
 |-- Aberto por: string (nullable = true)
 |-- Incidente Pai: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Entrou para KPI?: string (nullable = true)
 |-- KPI Violado?: string (nullable = true)



## Validação da Base

In [8]:
colunas_originais = df.columns
total_registros = df.count()

In [9]:
def identificar_nulos(df:DataFrame, nome_coluna: str) -> DataFrame:
    """
    Adiciona uma coluna booleana indicando se o valor na coluna informada é nulo.
    """
    return df.withColumn(
        f"{nome_coluna}_is_null", 
        F.col(nome_coluna).isNull()
    )

In [10]:
def identificar_valores_fora_do_padrao(df: DataFrame, nome_coluna: str, padrao: list) -> DataFrame:
    """
    Adiciona uma coluna booleana indicando se o valor na coluna informada está fora do padrão esperado.
    O parâmetro 'padrao' pode ser uma função ou expressão que define o critério de validação.
    """
    expressao_validacao = F.col(nome_coluna).isin(padrao)
    return df.withColumn(
        f"{nome_coluna}_is_out_of_pattern", 
        ~expressao_validacao
    )

In [11]:
# Criação de Coluna "Deve ser retirado da base"
df = df.withColumn(
    "Drop_from_base",
    F.lit(False)
)

In [12]:
filtro_registros = F.col("Drop_from_base") == False

### PK - Formato e Duplicatas

In [13]:
# Verificação de formato da coluna "Número" (PK)
"""
Valida se a coluna segue o padrão 'INC' seguido de exatamente 7 dígitos.
Exemplo: INC0001234
"""
# ^      : Início da string
# INC    : Prefixo literal
# \d{7}  : Exatamente 7 dígitos numéricos
# $      : Fim da string
padrao_regex = r"^INC\d{7}$"
df = df.withColumn(
    "Número_is_invalid", 
    ~F.col("Número").rlike(padrao_regex)
)
filtro = F.col("Número_is_invalid") == True
total_invalidos = df.filter(filtro).count()
pct_invalidos = (total_invalidos / total_registros)
if pct_invalidos > 0.005:
    print("⚠️ Atenção: Mais de 0.5% dos registros estão com 'Número' fora do padrão. Recomendado revisar a fonte dos dos dados.")
else:
    print("✅ A maioria dos registros segue o padrão esperado para 'Número'.")
    df = df.withColumn(
        "Drop_from_base", 
        F.when(F.col("Número_is_invalid"), True).otherwise(False)
    )
print(f"Total de registros com 'Número' fora do padrão: {total_invalidos} ({pct_invalidos*100:.2f}%)")
df.filter(filtro).show(total_invalidos, truncate=False)


✅ A maioria dos registros segue o padrão esperado para 'Número'.
Total de registros com 'Número' fora do padrão: 11 (0.01%)
+-------------------------------------------------------------------------+----------+-------------+----------+---------------+---------------+--------------------+------+---------+---------+-------+--------------------+------------------+-------+----------+-------------+------+----------------+------------+--------------+-----------------+
|Número                                                                   |Prioridade|Produto      |Categoria |Subcategoria   |Grupo designado|Item de configuração|Aberto|Resolvido|Encerrado|Duração|Código de fechamento|Descrição resumida|Solução|Aberto por|Incidente Pai|Status|Entrou para KPI?|KPI Violado?|Drop_from_base|Número_is_invalid|
+-------------------------------------------------------------------------+----------+-------------+----------+---------------+---------------+--------------------+------+---------+---------

In [14]:
total_registros = df.filter(filtro_registros).count()
total_registros

122543

In [15]:
# Verificação de Unicidade da PK - desconsiderando registros previamente marcados para remoção
chaves_distintas = df.filter(filtro_registros).select("Número").distinct().count()
unicidade_chave = total_registros == chaves_distintas
if unicidade_chave:
    print("✅ A coluna 'Número' é única.")
else:
    print("⚠️ A coluna 'Número' contém duplicatas.")
    print(f"Número de duplicatas: {total_registros - chaves_distintas} ({(total_registros - chaves_distintas)/total_registros*100:.2f}%)")
    # Adicionar uma coluna booleana indicando se o valor na coluna informada é duplicado.
    window_spec = Window.partitionBy("Número")
    df = df.withColumn(
        "Número_is_duplicate", 
        F.count("Número").over(window_spec) > 1
    )
    df.filter(F.col("Número_is_duplicate") == True).orderBy("Número").show(20)

✅ A coluna 'Número' é única.


### Regras de Negócio

In [ ]:
# Validando o status de cada incidente

In [18]:
df\
    .filter(filtro_registros)\
    .groupby("Status")\
    .count()\
    .orderBy(F.desc("count"))\
    .show(truncate=False)

+-------------------------+-----+
|Status                   |count|
+-------------------------+-----+
|Sem Intervenção          |80364|
|Encerrado Automaticamente|26830|
|Encerrado                |15337|
|NULL                     |11   |
|Aguardando Problema      |1    |
+-------------------------+-----+



In [ ]:
df\
    .filter(filtro_registros)\
    .filter(F.col("Status").isNotNull())\
    .groupby("Status")\
    .agg(
        F.count("*").alias("Total"), # Contagem total de registros por status
        F.sum(F.when(F.col("Resolvido").isNull(), 1).otherwise(0)).alias("Resolvido Nulo"), # Contagem de resolvidos nulos
        F.sum(F.when(F.col("Aberto por")=="Monitoramento", 1).otherwise(0)).alias("Abertos por Monitoramento") # Contagem abertos por monitoramento
    )\
    .orderBy(F.desc("Total"))\
    .show(truncate=False)

+-------------------------+-----+----------------+-------------------------+
|Status                   |Total|Resolvidos Nulos|Abertos por Monitoramento|
+-------------------------+-----+----------------+-------------------------+
|Sem Intervenção          |80364|80364           |80332                    |
|Encerrado Automaticamente|26830|1539            |15856                    |
|Encerrado                |15337|389             |8100                     |
|Aguardando Problema      |1    |1               |0                        |
+-------------------------+-----+----------------+-------------------------+



In [ ]:
# filtrando registros com status 'Sem Intervenção' e abertos manualmente

In [ ]:
# filtrando registros com status 'Encerrado' com resolvido nulo

In [ ]:
# filtrando registros com status 'Encerrado Automaticamente' com resolvido não nulo

In [ ]:
# colunas associadas ao status 
# sem intervenção - abertos por monitoramento, com resolvido nulo e grupo designidado nulo
# encerrado automaticamente - abertos por monitoramento, com resolvido nulo e grupo designidado não nulo
# encerrado -abertos por monitoramento, com resolvido não nulo e grupo designidado não nulo
# aguardando problema - só deus sabe

In [ ]:
# Validando os segundos de duração de um incidente

In [ ]:
# Validando o se todas as subcategorias preenchidas tem a categoria preenchida também

In [ ]:
# Validando se Entrou para KPI? atende as regras

# Regra 1: Somente as prioridades 1, 2 e 3 entram para o KPI
# Regra 2: Incidente Pai com valor preenchido não entram no KPI
# Regra 3: Status = “Sem Intervenção”, não entram no KPI

In [ ]:
# Validando se KPI Violado? atende as regras
# Regra 1 - Crítica - Duração até 4h
# Regra 2 - Alta - Duração até 4h
# Regra 3 - Média - Duração até 12h
# Regra 4 - Baixa - Duração até 24h
# Regra 5 - Muito Baixa - Duração até 96h

### Lista de Valores Permitidos

In [ ]:
# Validação de lista de valores permitidos por coluna
colunas_com_valores_definidos = {
    "Prioridade": ["1 - Crítica","2 - Alta","3 - Média","4 - Baixa","5 - Muito Baixa"],
    "Solução": ["Contorno", "Definitiva", ""],
    "Aberto por": ["Manual", "Monitoramento"],
    "Status": ["Aguardando Problema", "Encerrado", "Encerrado Automaticamente", "Sem Intervenção"],
    "Entrou para KPI?": ["SIM", "NAO"],
    "KPI Violado?": ["SIM", "NAO"]
}

### Nulos

In [43]:
# Validar valores nulos em colunas que não deveriam ter - desconsiderando registros previamente marcados para remoção
colunas_nao_nulas = [
    "Número",
    "Prioridade",
    "Grupo designado",
    "Aberto",
    "Encerrado",
    "Duração",
    "Descrição Resumida",
    "Aberto por",
    "Status",
    "Entrou para KPI?",
    "KPI Violado?"
]

for coluna in colunas_nao_nulas:
    df = identificar_nulos(df, coluna)

for coluna in colunas_nao_nulas:
    nulos_count = df.filter(filtro_registros).filter(F.col(f"{coluna}_is_null") == True).count()
    if nulos_count > 0:
        print(f"⚠️ A coluna '{coluna}' contém {nulos_count} valores nulos - ({nulos_count/total_registros*100:.2f}%).")
        df.filter(filtro_registros).filter(F.col(f"{coluna}_is_null") == True).select(colunas_originais).show(nulos_count, truncate=False)
    else:
        print(f"✅ A coluna '{coluna}' não contém valores nulos.")

✅ A coluna 'Número' não contém valores nulos.
✅ A coluna 'Prioridade' não contém valores nulos.
✅ A coluna 'Grupo designado' não contém valores nulos.
✅ A coluna 'Aberto' não contém valores nulos.
✅ A coluna 'Encerrado' não contém valores nulos.
✅ A coluna 'Duração' não contém valores nulos.
✅ A coluna 'Descrição Resumida' não contém valores nulos.
⚠️ A coluna 'Aberto por' contém 11 valores nulos - (0.01%).
+----------+----------+-------+---------+------------+---------------+--------------------+-------------------+-------------------+-------------------+-------+-----------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------+-------+----------+-------------+------+----------------+------------+
|Número    |Prioridade|Produto|Categoria|Subcategoria|Grupo designado|Item de configuração|Aberto             |Resolvido          |Encerrado          |Duração|Código de fechamento   |De

In [ ]:
# Aberto por
'''
Manual ou Monitoramento - Na Descrição Resumida, todos os registros que começam com "Problem: Alarm Application Monitoring" 
foram abertos por "Monitoramento". Portanto, é provável que os registros nulos tenham sido abertos da mesma forma, ou seja, por "Monitoramento".
'''
incidentes_similares = df\
    .filter(F.col("Descrição Resumida").startswith("Problem: Alarm Application Monitoring"))\
    .filter((F.col("Prioridade") != "4 - Baixa") & (F.col("Prioridade") != "5 - Muito Baixa"))\
    .filter(F.col("Aberto por").isNotNull())\
    .count()

incidentes_similares_monitoramento = df\
    .filter(F.col("Descrição Resumida").startswith("Problem: Alarm Application Monitoring"))\
    .filter((F.col("Prioridade") != "4 - Baixa") & (F.col("Prioridade") != "5 - Muito Baixa"))\
    .filter(F.col("Aberto por").isNotNull())\
    .filter(F.col("Aberto por")== "Monitoramento")\
    .count()

print(f"Total de incidentes com descrição e prioridade similar: {incidentes_similares}")
print(f"Total de incidentes com descrição e prioridade similar abertos por Monitoramento: {incidentes_similares_monitoramento}")
print(f"% de problemas similares abertos por Monitoramento: {(incidentes_similares_monitoramento/incidentes_similares)*100:.2f}%")

Total de incidentes com descrição e prioridade similar: 714
Total de incidentes com descrição e prioridade similar abertos por Monitoramento: 714
% de problemas similares abertos por Monitoramento: 100.00%


In [59]:
# Substituindo valores nulos em "Aberto por" por "Monitoramento", baseado na análise anterior
df = df.withColumn(
    "Aberto por",
    F.when(
        (F.col("Aberto por").isNull()),
        "Monitoramento"
    ).otherwise(F.col("Aberto por"))
)
df.filter(F.col("Aberto por_is_null")==True).select(colunas_originais).show()

+----------+----------+-------+---------+------------+---------------+--------------------+-------------------+-------------------+-------------------+-------+--------------------+--------------------+-------+-------------+-------------+------+----------------+------------+
|    Número|Prioridade|Produto|Categoria|Subcategoria|Grupo designado|Item de configuração|             Aberto|          Resolvido|          Encerrado|Duração|Código de fechamento|  Descrição resumida|Solução|   Aberto por|Incidente Pai|Status|Entrou para KPI?|KPI Violado?|
+----------+----------+-------+---------+------------+---------------+--------------------+-------------------+-------------------+-------------------+-------+--------------------+--------------------+-------+-------------+-------------+------+----------------+------------+
|INC8642461| 3 - Média|   NULL|     NULL|        NULL|         Team14|             IC00609|2025-12-21 23:16:19|               NULL|2025-12-21 23:28:51|    752|                

In [65]:
# Status
'''
Aguardando Problema
Encerrado
Encerrado Automaticamente
Sem Intervenção
'''
# Resolvido Nulo ou Não X Status "Sem Intervenção"
resolvido_nulo = df\
    .filter(F.col("Status") == "Sem Intervenção")\
    .filter(F.col("Resolvido").isNull())\
    .count()
resolvido_preenchido = df\
    .filter(F.col("Status") == "Sem Intervenção")\
    .filter(F.col("Resolvido").isNotNull())\
    .count()
total_registros_status = df.filter(F.col("Status") == "Sem Intervenção").count()
print(f"Total de registros com Status 'Sem Intervenção': {total_registros_status}")
print(f"Total de registros com Status 'Sem Intervenção' e 'Resolvido' nulo: {resolvido_nulo} ({(resolvido_nulo/total_registros_status)*100:.2f}%)")
print(f"Total de registros com Status 'Sem Intervenção' e 'Resolvido' preenchido: {resolvido_preenchido} ({(resolvido_preenchido/total_registros_status)*100:.2f}%)")

Total de registros com Status 'Sem Intervenção': 80364
Total de registros com Status 'Sem Intervenção' e 'Resolvido' nulo: 80364 (100.00%)
Total de registros com Status 'Sem Intervenção' e 'Resolvido' preenchido: 0 (0.00%)


In [66]:
# Preenchendo nulos de "Status", assumindo que os nulos do "Resolvido" estão corretos
df = df.withColumn(
    "Status",
    F.when(
        (F.col("Status").isNull()) & (F.col("Resolvido").isNull()),
        "Sem Intervenção"
    ).when(
        (F.col("Status").isNull()) & (F.col("Resolvido").isNotNull()),
        "Encerrado"
    ).otherwise(F.col("Status"))
)
df.filter(F.col("Status_is_null")==True).select(colunas_originais).show()


+----------+----------+-------+---------+------------+---------------+--------------------+-------------------+-------------------+-------------------+-------+--------------------+--------------------+-------+-------------+-------------+---------------+----------------+------------+
|    Número|Prioridade|Produto|Categoria|Subcategoria|Grupo designado|Item de configuração|             Aberto|          Resolvido|          Encerrado|Duração|Código de fechamento|  Descrição resumida|Solução|   Aberto por|Incidente Pai|         Status|Entrou para KPI?|KPI Violado?|
+----------+----------+-------+---------+------------+---------------+--------------------+-------------------+-------------------+-------------------+-------+--------------------+--------------------+-------+-------------+-------------+---------------+----------------+------------+
|INC8642461| 3 - Média|   NULL|     NULL|        NULL|         Team14|             IC00609|2025-12-21 23:16:19|               NULL|2025-12-21 23:28:

In [ ]:
# Entrou para KPI?
'''
Somente as prioridades 1, 2 e 3 entram para o KPI
Campo: Incidente Pai com valor preenchido não entram no KPI
Campo: Status = “Sem Intervenção”, não entram no KPI
'''

In [ ]:
# KPI Violado?
'''
1 - Crítica - Duração até 4h
2 - Alta - Duração até 4h
3 - Média - Duração até 12h
4 - Baixa - Duração até 24h
5 - Muito Baixa - Duração até 96h
'''